In [1]:
print("centralized_DL")

centralized_DL


In [2]:
import sys
print(sys.version)

3.10.20 (main, Mar  3 2026, 09:24:47) [GCC 13.3.0]


In [3]:
try:
    spark.stop()
except:
    pass

In [4]:
from pyspark.sql import SparkSession
spark= SparkSession.builder\
       .appName("Distributed_ML")\
       .getOrCreate()
spark
       
  


26/04/29 14:49:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
centralized_df= spark.read.parquet("/home/ubuntu/thesis/data/processed_data/system_state/centralized_data")
centralized_df.show(5)


+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|      Date|       value|            z_score|       lag_1|       lag_2|      Moving_avg|     Moving_std|       trend|          percentile|state_label|
+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+
|2015-08-01|1.27008217E8|-1.9898908848648427|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|
|2015-07-31|1.31522405E8|-1.8467134693292937|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|
|2015-07-25|1.33752964E8|-1.7759663775334804|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|
|2015-07-24|1.36192534E8| -1.698590041497396|1.43913454E8|1.49347436E8|1.431511413333E8|   661

In [6]:
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
label='state_label'

In [7]:
#normalizing the label into numbers
#0-Normal
#1 -degraded
#2 -failure
from pyspark.ml.feature import StringIndexer
indexer=StringIndexer(inputCol='state_label',outputCol='label')
C_df=indexer.fit(centralized_df).transform(centralized_df)
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']

C_df.select(*feature_columns,'label')

C_df.show()

+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|      Date|       value|            z_score|       lag_1|       lag_2|      Moving_avg|     Moving_std|       trend|          percentile|state_label|label|
+----------+------------+-------------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|2015-08-01|1.27008217E8|-1.9898908848648427|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|  0.0|
|2015-07-31|1.31522405E8|-1.8467134693292937|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|  0.0|
|2015-07-25|1.33752964E8|-1.7759663775334804|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|  0.0|
|2015-07-24|1.36192534E8| -1.698590041497396|1.43913454E8|

In [8]:
C_df_new= C_df.drop("z_score")
C_df_new.show(5)

[Stage 6:>                                                          (0 + 1) / 1]

+----------+------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|      Date|       value|       lag_1|       lag_2|      Moving_avg|     Moving_std|       trend|          percentile|state_label|label|
+----------+------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|2015-08-01|1.27008217E8|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|  0.0|
|2015-07-31|1.31522405E8|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|  0.0|
|2015-07-25|1.33752964E8|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|  0.0|
|2015-07-24|1.36192534E8|1.43913454E8|1.49347436E8|1.431511413333E8|   6610499.3842|  -7720920.0| 0.00546448087431694|     Normal|  0.0|
|2015-07-30|1.36471572E8|1.61441517E8|1.4

In [9]:
from pyspark.sql.functions import col, isnan, when, count
C_df_new.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in C_df_new.columns
]).show()

[Stage 7:>                                                          (0 + 1) / 1]

+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+
|Date|value|lag_1|lag_2|Moving_avg|Moving_std|trend|percentile|state_label|label|
+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+
|   0|    0|    1|    2|         0|         1|    1|         0|          0|    0|
+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+



In [10]:
from pyspark.sql.functions import col
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
C_df_new=C_df_new.dropna(subset=feature_columns)

In [11]:
# after removing the NUll valeus 

from pyspark.sql.functions import col, isnan, when, count
C_df_new.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in C_df_new.columns
]).show()

[Stage 10:>                                                         (0 + 1) / 1]

+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+
|Date|value|lag_1|lag_2|Moving_avg|Moving_std|trend|percentile|state_label|label|
+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+
|   0|    0|    0|    0|         0|         0|    0|         0|          0|    0|
+----+-----+-----+-----+----------+----------+-----+----------+-----------+-----+



In [12]:
C_df_clean=C_df_new
C_df_clean.show(5)

[Stage 13:>                                                         (0 + 1) / 1]

+----------+------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|      Date|       value|       lag_1|       lag_2|      Moving_avg|     Moving_std|       trend|          percentile|state_label|label|
+----------+------------+------------+------------+----------------+---------------+------------+--------------------+-----------+-----+
|2015-08-01|1.27008217E8|1.31522405E8|1.36471572E8|    1.31667398E8|   4733343.3418|  -4514188.0|                 0.0|     Normal|  0.0|
|2015-07-31|1.31522405E8|1.36471572E8|1.61441517E8|1.431451646667E8|1.60371738479E7|  -4949167.0|0.001821493624772...|     Normal|  0.0|
|2015-07-25|1.33752964E8|1.36192534E8|1.43913454E8|    1.37952984E8|   5304081.2034|  -2439570.0|0.003642987249544...|     Normal|  0.0|
|2015-07-24|1.36192534E8|1.43913454E8|1.49347436E8|1.431511413333E8|   6610499.3842|  -7720920.0| 0.00546448087431694|     Normal|  0.0|
|2015-07-30|1.36471572E8|1.61441517E8|1.4

In [13]:
C_df_clean.groupBy("label").count().show()

[Stage 14:>                                                         (0 + 1) / 1]

+-----+-----+
|label|count|
+-----+-----+
|  0.0|  218|
|  1.0|  192|
|  2.0|  138|
+-----+-----+



In [14]:
from pyspark.ml.feature import VectorAssembler
feature_columns=['value','Moving_avg','Moving_std','lag_1','lag_2','trend']
assembler=VectorAssembler(
    inputCols=feature_columns,
    outputCol="features")


C_df_model=assembler.transform(C_df_clean)
C_df_model.count()

train = C_df_model.filter(col("Date") < "2016-06-01")
test  = C_df_model.filter(col("Date") >= "2016-06-01")

print("Train:", train.count())
print("Test:", test.count())

Train: 334
Test: 214


In [ ]:


from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col


scaler = StandardScaler(
    inputCol="features",
    outputCol="scaledFeatures",
    withMean=True,
    withStd=True
)



input_size = len(feature_columns)

# Count distinct labels
num_classes = C_df_model.select("label").distinct().count()

layers = [
    input_size,   # Input layer
    16,           # Hidden layer 1
    8,            # Hidden layer 2
    num_classes   # Output layer
]


ann = MultilayerPerceptronClassifier(
    featuresCol="scaledFeatures",
    labelCol="label",
    layers=layers,
    maxIter=100,
    blockSize=128,
    stepSize=0.03,
    seed=42
)


pipeline = Pipeline(stages=[scaler, ann])


ann_model = pipeline.fit(train)


predictions = ann_model.transform(test)


accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

precision_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedPrecision"
)

recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)

accuracy = accuracy_evaluator.evaluate(predictions)
f1_score = f1_evaluator.evaluate(predictions)
precision = precision_evaluator.evaluate(predictions)
recall = recall_evaluator.evaluate(predictions)


print("===== ANN Centralized Model Performance =====")
print(f"Accuracy  : {accuracy:.4f}")
print(f"F1 Score  : {f1_score:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


predictions.select("Date", "label", "prediction", "probability").show(10, truncate=False)

26/04/29 14:55:30 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
                                                                                

===== ANN Centralized Model Performance =====
Accuracy  : 0.9673
F1 Score  : 0.9674
Precision : 0.9679
Recall    : 0.9673
+----------+-----+----------+----------------------------------------------------------------+
|Date      |label|prediction|probability                                                     |
+----------+-----+----------+----------------------------------------------------------------+
|2016-06-03|0.0  |0.0       |[0.9999999973793818,2.620618098737478E-9,4.00586943986089E-32]  |
|2016-07-08|0.0  |0.0       |[0.9999999963946653,3.605334724678036E-9,7.209390533611947E-32] |
|2016-07-15|0.0  |0.0       |[0.9999999973008702,2.699129906872655E-9,4.228383109260101E-32] |
|2016-07-16|0.0  |0.0       |[0.9999999971360884,2.863911575185417E-9,4.7159945187040036E-32]|
|2016-07-09|0.0  |0.0       |[0.9999999964436794,3.556320595160775E-9,7.01186066840139E-32]  |
|2016-06-18|0.0  |0.0       |[0.9999999978128755,2.1871245393795473E-9,2.846847174468004E-32]|
|2016-07-14|0.0  |0.0  